# 02 — Basic Network Statistics

This notebook computes the first global statistics of the `cit-HepTh` citation network.

The goal is to describe the overall structure of the directed graph before moving to degree distributions, centrality measures, ranking comparison, community detection, and temporal evolution.

At this stage, we focus on:

- loading the cleaned edge list created in Notebook 01;
- counting nodes and edges;
- checking graph density;
- identifying weakly and strongly connected components;
- computing reciprocity;
- preparing a compact summary table.

No centrality analysis is performed in this notebook.

## 1. Imports and project paths

In [1]:
from pathlib import Path

import pandas as pd
import networkx as nx

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

PROCESSED_EDGES_PATH = PROCESSED_DIR / "cit_hepth_clean_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Current working directory: {CURRENT_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Processed edges path: {PROCESSED_EDGES_PATH}")
print(f"Processed edges file exists: {PROCESSED_EDGES_PATH.exists()}")

Current working directory: /Users/chiarabelli/Desktop/citation-network-project/notebooks
Project root: /Users/chiarabelli/Desktop/citation-network-project
Processed edges path: /Users/chiarabelli/Desktop/citation-network-project/data/processed/cit_hepth_clean_edges.parquet
Processed edges file exists: True


## 2. Start Spark session

The cleaned edge list is stored in Parquet format after Notebook 01.  
Spark is used here to load the data and compute the first scalable aggregations.

In [3]:
spark = (
    SparkSession.builder
    .appName("cit-hepth-basic-network-statistics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/15 11:57:56 WARN Utils: Your hostname, MacBook-Pro-di-Chiara.local, resolves to a loopback address: 127.0.0.1; using 10.208.162.29 instead (on interface en0)
26/05/15 11:57:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 11:57:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Load cleaned edge list

The edge list should contain two columns:

- `from_node_id`: citing paper;
- `to_node_id`: cited paper.

In [4]:
edges_sdf = spark.read.parquet(str(PROCESSED_EDGES_PATH))

edges_sdf.printSchema()

root
 |-- from_node_id: long (nullable = true)
 |-- to_node_id: long (nullable = true)



In [5]:
edges_sdf.show(10, truncate=False)

+------------+----------+
|from_node_id|to_node_id|
+------------+----------+
|1001        |9503124   |
|1001        |9505162   |
|1001        |9510225   |
|1001        |9511171   |
|1001        |9602114   |
|1001        |9701162   |
|1001        |9702198   |
|1001        |9705030   |
|1001        |9907041   |
|1001        |9908144   |
+------------+----------+
only showing top 10 rows


## 4. Basic size statistics

In a directed citation network:

- each node is a paper;
- each edge is a citation from one paper to another;
- `from_node_id -> to_node_id` means that the source paper cites the target paper.

In [6]:
num_edges = edges_sdf.count()

source_nodes_sdf = edges_sdf.select(F.col("from_node_id").alias("node_id"))
target_nodes_sdf = edges_sdf.select(F.col("to_node_id").alias("node_id"))
nodes_sdf = source_nodes_sdf.union(target_nodes_sdf).distinct()

num_nodes = nodes_sdf.count()

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of edges: {num_edges:,}")

Number of nodes: 27,769
Number of edges: 352,768


## 5. Directed graph density

For a directed graph without self-loops, density is defined as:

$$
\text{Density} = \frac{m}{n(n-1)}
$$

where:

- $m$ is the number of directed edges;
- $n$ is the number of nodes.

Citation networks are usually sparse: the number of possible citations is much larger than the number of observed citations.

In [7]:
if num_nodes > 1:
    directed_density = num_edges / (num_nodes * (num_nodes - 1))
else:
    directed_density = 0

print(f"Directed graph density: {directed_density:.8f}")

Directed graph density: 0.00045749


It is a very small value, and it is normal. A citation network is **sparse**: each paper cites only a limited portion of relevant papers. 

## 6. Convert to NetworkX for component-level statistics

PySpark is useful for scalable preprocessing and aggregation, but NetworkX is convenient for graph-level structural statistics.

The full `cit-HepTh` graph is small enough to be handled in memory for this first global analysis.

In [8]:
edges_pdf = edges_sdf.toPandas()

G_directed = nx.from_pandas_edgelist(
    edges_pdf,
    source="from_node_id",
    target="to_node_id",
    create_using=nx.DiGraph()
)

print(G_directed)

DiGraph with 27769 nodes and 352768 edges


## 7. Connected components

For directed graphs, we distinguish between:

- **weakly connected components**: components obtained by ignoring edge direction;
- **strongly connected components**: groups of nodes where every node can reach every other node following edge directions.

In citation networks, strongly connected components are often small because citations usually point backward in time.

In [9]:
weak_components = list(nx.weakly_connected_components(G_directed))
strong_components = list(nx.strongly_connected_components(G_directed))

num_weak_components = len(weak_components)
num_strong_components = len(strong_components)

largest_weak_component_size = max(len(component) for component in weak_components)
largest_strong_component_size = max(len(component) for component in strong_components)

weak_component_ratio = largest_weak_component_size / num_nodes
strong_component_ratio = largest_strong_component_size / num_nodes

print(f"Number of weakly connected components: {num_weak_components:,}")
print(f"Largest weakly connected component size: {largest_weak_component_size:,}")
print(f"Largest weakly connected component ratio: {weak_component_ratio:.4f}")

print()

print(f"Number of strongly connected components: {num_strong_components:,}")
print(f"Largest strongly connected component size: {largest_strong_component_size:,}")
print(f"Largest strongly connected component ratio: {strong_component_ratio:.4f}")

Number of weakly connected components: 142
Largest weakly connected component size: 27,400
Largest weakly connected component ratio: 0.9867

Number of strongly connected components: 20,085
Largest strongly connected component size: 7,464
Largest strongly connected component ratio: 0.2688


## 8. Reciprocity

Reciprocity measures the fraction of directed edges that are reciprocated.

In this project, a reciprocal pair would mean that paper A cites paper B and paper B cites paper A.  
This should be rare in a citation network, but it is still useful to check because metadata, versions, or data inconsistencies may create cycles.

In [13]:
reciprocity = nx.reciprocity(G_directed)

print(f"Graph reciprocity: {reciprocity:.8f}" if reciprocity is not None else "Graph reciprocity: undefined")

Graph reciprocity: 0.00273834


## 9. Undirected projection statistics

Some network measures are easier to interpret on the undirected projection of the graph.

Here, edge direction is ignored.  
This is useful for component size, clustering, and later community detection.

In [14]:
G_undirected = G_directed.to_undirected()

num_undirected_edges = G_undirected.number_of_edges()

if num_nodes > 1:
    undirected_density = num_undirected_edges / (num_nodes * (num_nodes - 1) / 2)
else:
    undirected_density = 0

average_clustering = nx.average_clustering(G_undirected)

print(G_undirected)
print(f"Undirected edges: {num_undirected_edges:,}")
print(f"Undirected density: {undirected_density:.8f}")
print(f"Average clustering coefficient: {average_clustering:.8f}")

Graph with 27769 nodes and 352285 edges
Undirected edges: 352,285
Undirected density: 0.00091373
Average clustering coefficient: 0.31203073


## 10. Summary table

This table summarizes the first structural properties of the network.

In [15]:
summary_rows = [
    ("Nodes", num_nodes),
    ("Directed edges", num_edges),
    ("Directed density", directed_density),
    ("Weakly connected components", num_weak_components),
    ("Largest weak component size", largest_weak_component_size),
    ("Largest weak component ratio", weak_component_ratio),
    ("Strongly connected components", num_strong_components),
    ("Largest strong component size", largest_strong_component_size),
    ("Largest strong component ratio", strong_component_ratio),
    ("Reciprocity", reciprocity if reciprocity is not None else 0),
    ("Undirected edges", num_undirected_edges),
    ("Undirected density", undirected_density),
    ("Average clustering coefficient", average_clustering),
]

summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Value"])

summary_df

,Metric,Value
0,Nodes,27769.000000
1,Directed edges,352768.000000
2,Directed density,0.000457
3,Weakly connected components,142.000000
4,Largest weak component size,27400.000000
5,Largest weak component ratio,0.986712
6,Strongly connected components,20085.000000
7,Largest strong component size,7464.000000
8,Largest strong component ratio,0.268789
9,Reciprocity,0.002738


## 11. Save summary output

The summary table is saved as a CSV file so that it can be reused in the final report.

In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_output_path = OUTPUT_DIR / "02_basic_network_statistics_summary.csv"
summary_df.to_csv(summary_output_path, index=False)

print(f"Summary saved to: {summary_output_path}")

Summary saved to: /Users/chiarabelli/Desktop/citation-network-project/outputs/02_basic_network_statistics_summary.csv


## 12. Stop Spark session

In [17]:
spark.stop()